# Market Data Analysis with Alpaca SDK

This notebook demonstrates how to use the Alpaca Market Data SDK for analyzing market data, creating visualizations, and building trading insights.

## Prerequisites

- Install required packages: `pip install alpaca-market-data pandas matplotlib jupyter`
- Set up your `.env` file with API credentials

## Setup

In [ ]:
# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import numpy as np
from alpaca_data import AlpacaClient

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Initialize client
client = AlpacaClient()

print("Alpaca Market Data SDK - Market Analysis Notebook")
print("=" * 50)

## 1. Basic Data Collection

In [ ]:
# Get historical data for multiple stocks
symbols = ['AAPL', 'GOOGL', 'MSFT', 'TSLA']

# Get 3 months of daily data
end_date = datetime.now()
start_date = end_date - timedelta(days=90)

print(f"Collecting data from {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
print(f"Symbols: {', '.join(symbols)}")

# Get data for all symbols
data = client.get_bars(
    symbols=symbols,
    timeframe='1Day',
    start=start_date.isoformat(),
    limit=90
)

print(f"Retrieved {len(data['bars'])} total bars")
print(f"Symbol breakdown:")
for symbol in symbols:
    count = len([bar for bar in data['bars'] if bar.symbol == symbol])
    print(f"  {symbol}: {count} bars")

## 2. Data Preparation

In [ ]:
# Convert to DataFrame for analysis
df_data = []
for bar in data['bars']:
    df_data.append({
        'symbol': bar.symbol,
        'timestamp': bar.timestamp,
        'open': bar.open,
        'high': bar.high,
        'low': bar.low,
        'close': bar.close,
        'volume': bar.volume
    })

df = pd.DataFrame(df_data)

# Add derived indicators
df['daily_return'] = df.groupby('symbol')['close'].pct_change()
df['volatility_20'] = df.groupby('symbol')['daily_return'].rolling(20).std()
df['sma_20'] = df.groupby('symbol')['close'].rolling(20).mean()
df['price_change'] = df.groupby('symbol')['close'].pct_change()

print("DataFrame created with derived indicators")
print(f"Shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"\nFirst few rows:")
print(df.head())

## 3. Price Analysis and Visualization

In [ ]:
# Create price comparison plot
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Market Analysis Dashboard', fontsize=16)

# Plot 1: Price trends
ax1 = axes[0, 0]
for symbol in symbols:
    symbol_data = df[df['symbol'] == symbol]
    ax1.plot(symbol_data['timestamp'], symbol_data['close'], label=symbol, linewidth=2)

ax1.set_title('Price Trends')
ax1.set_xlabel('Date')
ax1.set_ylabel('Price ($)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Volume comparison
ax2 = axes[0, 1]
for symbol in symbols:
    symbol_data = df[df['symbol'] == symbol]
    ax2.plot(symbol_data['timestamp'], symbol_data['volume'], label=symbol, alpha=0.7)

ax2.set_title('Volume Trends')
ax2.set_xlabel('Date')
ax2.set_ylabel('Volume')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Daily returns distribution
ax3 = axes[1, 0]
for symbol in symbols:
    symbol_data = df[df['symbol'] == symbol]
    ax3.hist(symbol_data['daily_return'].dropna(), bins=30, alpha=0.6, label=symbol, density=True)

ax3.set_title('Daily Returns Distribution')
ax3.set_xlabel('Daily Return')
ax3.set_ylabel('Density')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Volatility comparison
ax4 = axes[1, 1]
for symbol in symbols:
    symbol_data = df[df['symbol'] == symbol]
    ax4.plot(symbol_data['timestamp'], symbol_data['volatility_20'], label=f'{symbol} Vol', linewidth=2)

ax4.set_title('20-Day Rolling Volatility')
ax4.set_xlabel('Date')
ax4.set_ylabel('Volatility')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Statistical Analysis

In [ ]:
# Calculate key statistics
print("Statistical Summary")
print("=" * 40)

for symbol in symbols:
    symbol_data = df[df['symbol'] == symbol]
    returns = symbol_data['daily_return'].dropna()
    
    print(f"\n{symbol}:")
    print(f"  Price range: ${symbol_data['close'].min():.2f} - ${symbol_data['close'].max():.2f}")
    print(f"  Average daily return: {returns.mean()*100:.2f}%")
    print(f"  Daily volatility: {returns.std()*100:.2f}%")
    print(f"  Annualized volatility: {returns.std()*np.sqrt(252)*100:.1f}%")
    print(f"  Sharpe ratio (risk-free=0%): {returns.mean()/returns.std()*np.sqrt(252):.2f}")

# Correlation analysis
print("\n" + "="*40)
print("Correlation Analysis")
print("="*40)

# Create correlation matrix
pivot_df = df.pivot(index='timestamp', columns='symbol', values='daily_return')
correlation_matrix = pivot_df.corr()

print("\nDaily Returns Correlation Matrix:")
print(correlation_matrix.round(3))

# Visualize correlation
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.3f')
plt.title('Stock Returns Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Technical Indicators

In [ ]:
# Calculate more technical indicators
def calculate_technical_indicators(df, symbol):
    """Calculate technical indicators for a symbol."""
    symbol_data = df[df['symbol'] == symbol].copy()
    symbol_data = symbol_data.sort_values('timestamp')
    
    # Simple Moving Averages
    symbol_data['sma_5'] = symbol_data['close'].rolling(5).mean()
    symbol_data['sma_20'] = symbol_data['close'].rolling(20).mean()
    symbol_data['sma_50'] = symbol_data['close'].rolling(50).mean()
    
    # RSI calculation
    delta = symbol_data['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    symbol_data['rsi'] = 100 - (100 / (1 + rs))
    
    # Bollinger Bands
    symbol_data['bb_middle'] = symbol_data['close'].rolling(20).mean()
    symbol_data['bb_std'] = symbol_data['close'].rolling(20).std()
    symbol_data['bb_upper'] = symbol_data['bb_middle'] + (symbol_data['bb_std'] * 2)
    symbol_data['bb_lower'] = symbol_data['bb_middle'] - (symbol_data['bb_std'] * 2)
    
    return symbol_data

# Analyze a specific stock (e.g., AAPL)
symbol = 'AAPL'
technical_data = calculate_technical_indicators(df, symbol)

print(f"Technical Analysis for {symbol}")
print("="*40)

latest = technical_data.iloc[-1]
print(f"Current price: ${latest['close']:.2f}")
print(f"5-day SMA: ${latest['sma_5']:.2f}")
print(f"20-day SMA: ${latest['sma_20']:.2f}")
print(f"RSI (14): {latest['rsi']:.2f}")
print(f"Bollinger Bands: ${latest['bb_lower']:.2f} - ${latest['bb_upper']:.2f}")

# Plot technical indicators
fig, axes = plt.subplots(3, 1, figsize=(12, 10))
fig.suptitle(f'{symbol} Technical Analysis', fontsize=16)

# Price and moving averages
ax1 = axes[0]
ax1.plot(technical_data['timestamp'], technical_data['close'], label='Price', linewidth=2)
ax1.plot(technical_data['timestamp'], technical_data['sma_5'], label='5-day SMA', alpha=0.7)
ax1.plot(technical_data['timestamp'], technical_data['sma_20'], label='20-day SMA', alpha=0.7)
ax1.plot(technical_data['timestamp'], technical_data['sma_50'], label='50-day SMA', alpha=0.7)
ax1.fill_between(technical_data['timestamp'], technical_data['bb_lower'], technical_data['bb_upper'], 
                 alpha=0.2, label='Bollinger Bands')
ax1.set_title('Price and Moving Averages')
ax1.set_ylabel('Price ($)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Volume
ax2 = axes[1]
ax2.bar(technical_data['timestamp'], technical_data['volume'], alpha=0.6)
ax2.set_title('Volume')
ax2.set_ylabel('Volume')
ax2.grid(True, alpha=0.3)

# RSI
ax3 = axes[2]
ax3.plot(technical_data['timestamp'], technical_data['rsi'], color='purple', linewidth=2)
ax3.axhline(y=70, color='r', linestyle='--', alpha=0.7, label='Overbought (70)')
ax3.axhline(y=30, color='g', linestyle='--', alpha=0.7, label='Oversold (30)')
ax3.set_title('RSI (14-day)')
ax3.set_xlabel('Date')
ax3.set_ylabel('RSI')
ax3.set_ylim(0, 100)
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Real-time Data Monitoring

In [ ]:
# Get current market snapshots
print("Current Market Snapshots")
print("="*40)

snapshots = client.get_snapshot(symbols=symbols)

for snapshot in snapshots['snapshots']:
    print(f"\n{snapshot.symbol}:")
    
    if snapshot.latest_trade:
        trade = snapshot.latest_trade
        print(f"  Latest Trade: ${trade.price:.2f} x {trade.size} @ {trade.timestamp}")
    
    if snapshot.latest_quote:
        quote = snapshot.latest_quote
        spread = quote.ask_price - quote.bid_price
        print(f"  Quote: ${quote.bid_price:.2f} x {quote.bid_size} / ${quote.ask_price:.2f} x {quote.ask_size}")
        print(f"  Spread: ${spread:.4f}")
    
    if snapshot.daily_bar:
        bar = snapshot.daily_bar
        change = ((bar.close - bar.open) / bar.open) * 100
        print(f"  Daily Bar: O:${bar.open:.2f} H:${bar.high:.2f} L:${bar.low:.2f} C:${bar.close:.2f} ({change:+.2f}%)")
        print(f"  Volume: {bar.volume:,}")

## 7. Cryptocurrency Analysis

In [ ]:
# Get cryptocurrency data
crypto_symbols = ['BTC/USD', 'ETH/USD']

print("Cryptocurrency Analysis")
print("="*40)

# Get crypto bars
crypto_bars = client.get_crypto_bars(
    symbols=crypto_symbols,
    timeframe='1Day',
    start=(end_date - timedelta(days=30)).isoformat(),
    limit=30
)

# Convert to DataFrame
crypto_df_data = []
for bar in crypto_bars['bars']:
    crypto_df_data.append({
        'symbol': bar.symbol,
        'timestamp': bar.timestamp,
        'open': bar.open,
        'high': bar.high,
        'low': bar.low,
        'close': bar.close,
        'volume': bar.volume
    })

crypto_df = pd.DataFrame(crypto_df_data)
crypto_df['daily_return'] = crypto_df.groupby('symbol')['close'].pct_change()

# Plot crypto comparison
plt.figure(figsize=(12, 6))
for symbol in crypto_symbols:
    symbol_data = crypto_df[crypto_df['symbol'] == symbol]
    plt.plot(symbol_data['timestamp'], symbol_data['close'], label=symbol, linewidth=2)

plt.title('Cryptocurrency Price Comparison (30 Days)')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')  # Log scale for better visualization
plt.tight_layout()
plt.show()

# Crypto statistics
print("\nCryptocurrency Statistics:")
for symbol in crypto_symbols:
    symbol_data = crypto_df[crypto_df['symbol'] == symbol]
    returns = symbol_data['daily_return'].dropna()
    
    print(f"\n{symbol}:")
    print(f"  Current price: ${symbol_data['close'].iloc[-1]:.2f}")
    print(f"  30-day range: ${symbol_data['close'].min():.2f} - ${symbol_data['close'].max():.2f}")
    print(f"  Average daily return: {returns.mean()*100:.2f}%")
    print(f"  Daily volatility: {returns.std()*100:.2f}%")

## 8. Summary and Insights

This notebook demonstrates the power of the Alpaca Market Data SDK for:

- Historical data analysis
- Real-time market monitoring
- Technical analysis and indicators
- Multi-asset comparison
- Statistical analysis and visualization
- Cryptocurrency data integration

The SDK provides comprehensive market data access with clean, Pythonic interfaces suitable for both beginner and advanced analysts.

In [ ]:
# Final summary
print("\n" + "="*60)
print("MARKET ANALYSIS SUMMARY")
print("="*60)

# Stock summary
print("\nSTOCKS ANALYZED:")
for symbol in symbols:
    symbol_data = df[df['symbol'] == symbol]
    current_price = symbol_data['close'].iloc[-1]
    period_return = ((current_price - symbol_data['close'].iloc[0]) / symbol_data['close'].iloc[0]) * 100
    
    print(f"  {symbol}: ${current_price:.2f} ({period_return:+.1f}% over period)")

# Crypto summary
print("\nCRYPTOCURRENCIES ANALYZED:")
for symbol in crypto_symbols:
    symbol_data = crypto_df[crypto_df['symbol'] == symbol]
    current_price = symbol_data['close'].iloc[-1]
    period_return = ((current_price - symbol_data['close'].iloc[0]) / symbol_data['close'].iloc[0]) * 100
    
    print(f"  {symbol}: ${current_price:.2f} ({period_return:+.1f}% over period)")

print("\n" + "="*60)
print("Analysis complete! Data collected using Alpaca Market Data SDK")
print("="*60)